# 03 · Reproducing parts of the paper

Scaled-down reproductions of the paper's main figures, on the **surface code**:

1. **Fig. 3** — non-local CNOT vs teleportation as the physical error rate varies,
   at two ebit-noise levels.
2. **Fig. 4** — non-local CNOT as ebit **decoherence** (wait-time noise) grows.
3. **Fig. 5** — the Pauli-product-measurement compute step vs Pauli-string weight.

> **Surface codes only here.** The bivariate-bicycle (qLDPC) codes are slow to
> decode single-threaded with the Tesseract decoder — a single low-`p` BB point can
> take minutes — so this laptop-scale notebook sticks to the `[[9,1,3]]` surface
> code. To reproduce the **BB** curves and larger distances at scale, use the
> MPI-parallel scripts in [`scripts/`](../scripts) (see `scripts/README.md`).

> **These are laptop-scale runs.** We use the smallest code and only enough shots
> to see the trends (`TARGET_ERRORS` below). The published figures used distance up
> to 11–12 and billions of shots across a cluster. Increase `TARGET_ERRORS` and add
> lower error rates for smoother curves.


In [ ]:
# --- make the tmcbs package importable whether this notebook is launched from
# --- the package directory or from notebooks/ ------------------------------
import os, sys
for _cand in (os.getcwd(), os.path.abspath(os.path.join(os.getcwd(), ".."))):
    if os.path.isdir(os.path.join(_cand, "tmcbs")) and _cand not in sys.path:
        sys.path.insert(0, _cand)

import numpy as np
import matplotlib.pyplot as plt
import tmcbs as t
print("tmcbs version", t.__version__)

# "teser" (Tesseract) is the decoder behind the paper's main results. If the
# compiled tesseract_decoder wheel will not load on your machine, switch to a
# pure-Python fallback: "BP-OSD" or "LSD".
DECODER = "teser"

TARGET_ERRORS = 30        # raise for smoother curves (and longer runtime)

def sweep(make_circuit, xs, p_of_x, d):
    """Estimate LER over a sweep. make_circuit(x)->circuit, p_of_x(x)->phys rate."""
    ler, lo, hi = [], [], []
    for x in xs:
        r = t.estimate_ler(make_circuit(x), p=p_of_x(x), d=d, decoder=DECODER,
                           target_errors=TARGET_ERRORS, max_shots=5_000_000)
        ler.append(r.ler); lo.append(r.ler - r.ci_low); hi.append(r.ci_high - r.ler)
        print(f"    x={x!s:<10} LER={r.ler:.2e}  ({r.errors} err / {r.shots} shots)")
    return np.array(ler), np.array([lo, hi])


## Fig. 3 — ebit noise: non-local CNOT vs teleportation

Sweep the physical error rate `p` for the `[[9,1,3]]` surface code and decode both
primitives, at ebit noise equal to `p` and at `10p`. The non-local CNOT stays below
teleportation at matched parameters, and ebit noise 10x the physical rate is still
tolerated.


In [ ]:
code = t.surface_code(5)
ps = [1e-2, 5e-3, 3e-3, 1e-3]

fig, ax = plt.subplots(figsize=(6.6, 4.6))
for ratio, style in [(1, "-"), (10, ":")]:          # 10x-PER ebit noise dotted
    for prim, marker, col in [("CNOT", "o", "C0"), ("TELE", "s", "C3")]:
        print(f"  {code.name} {prim} ebit={ratio}xPER")
        mk = (lambda p, prim=prim, ratio=ratio:
              t.non_local_cnot(code, p, min(ratio * p, 1.0)) if prim == "CNOT"
              else t.teleportation(code, p, min(ratio * p, 1.0)))
        ler, err = sweep(mk, ps, lambda p: p, code.d)
        ax.errorbar(ps, ler, yerr=err, marker=marker, ls=style, color=col,
                    capsize=2, label=f"{prim}, ebit={ratio}xPER")
ax.plot(ps, ps, "k:", lw=1, label="break-even")
ax.set(xscale="log", yscale="log", title=code.name,
       xlabel="physical error rate p", ylabel="logical error rate")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()


## Fig. 4 — ebit decoherence

Now hold `p = p_ebit = 1e-3` fixed and grow the **decoherence ratio** (oldest-ebit
wait time / T1) applied to the ebits before the non-local CNOT. We compare the two
production schedules: one-at-a-time generation vs `O(d)` "line" generation
(`ebits_per_cycle = d`). The LER stays flat until the wait-time noise becomes
comparable to the physical error rate, then rises.


In [ ]:
code = t.surface_code(5)
ratios = [1e-3, 1e-2, 1e-1, 5e-1, 1e0]

fig, ax = plt.subplots(figsize=(6.4, 4.4))
for per_cycle, style, lab in [(-1, "-", "one-at-a-time"), (code.d, "--", "line O(d)")]:
    print(f"  {code.name} {lab}")
    mk = lambda r, pc=per_cycle: t.non_local_cnot(
        code, 1e-3, 1e-3, decoherence_ratio=r, ebits_per_cycle=pc)
    ler, err = sweep(mk, ratios, lambda r: 1e-3, code.d)
    ax.errorbar(ratios, ler, yerr=err, marker="o", ls=style, color="C0",
                capsize=2, label=lab)
ax.set(xscale="log", yscale="log", xlabel="ebit decoherence ratio (wait time / T1)",
       ylabel="non-local CNOT LER",
       title=f"{code.name}: ebit decoherence (p = p_ebit = 1e-3)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()


## Fig. 5 — Pauli-product-measurement compute step

The PPM compute step is a chain of transversal CNOTs onto an ancilla (paper
Methods, "Pauli string compute"). A per-link pattern string sets which links are
local (`0`) vs non-local (`1`); a pattern of length `L` realises a weight
`w = L + 1` Pauli string. We sweep the weight for the surface code.


In [ ]:
code = t.surface_code(5)
patterns = ["1", "10", "101"]          # weights 2, 3, 4
weights = [len(p) + 1 for p in patterns]
P = 1e-3

print(f"  {code.name}")
mk = lambda pat: t.pauli_product_measurement(code, P, P, pat)
ler, err = sweep(mk, patterns, lambda pat: P, code.d)

fig, ax = plt.subplots(figsize=(6.4, 4.4))
ax.errorbar(weights, ler, yerr=err, marker="o", color="C0", capsize=2, label=code.name)
ax.set(yscale="log", xlabel="Pauli string weight w", ylabel="compute-step LER",
       title=f"PPM compute step (p = p_ebit = {P:g})")
ax.set_xticks(weights); ax.legend(fontsize=8); plt.tight_layout(); plt.show()


## Scaling up — BB codes, larger distances, the cluster

The curves above are coarse, surface-code-only previews. The bivariate-bicycle
codes and the paper's regime (distance 11–12, LER ≤ 1e-10) need many shots and the
slow BB decode parallelised over MPI ranks. The same primitives drive the MPI
runner; for example a single non-local-CNOT BB sweep:

```bash
mpirun -n 256 python -m tmcbs.run_ebit_experiment_mpi --experiment 1 \
    --bicycle-code --l 12 --m 6 --Ax 3 --Ay 1,2 --Bx 1,2 --By 3 \
    --distance 12 --n 144 --phys-noise 1e-3,5e-4,1e-4 --trans-ratio 10 \
    --file-name cx_bb_d12
```

The ready-made reproduction scripts (including the BB versions of Figs 3–5) live in
[`scripts/`](../scripts). Each writes an `.npz` with `resultsArr`,
`totalErrorsArr`, `numberOfShotsArr`; load it and fit Bayes-factor error bars with
`t.bayes_factor_interval(errors, shots)` exactly as `estimate_ler` does here.
